# 4.0 — Baselines clasificación multiclase (nonMalignant vs tipos de cáncer)

Este notebook replica el flujo de la iteración binaria, pero con etiquetas multiclase:
- `nonMalignant` (todas las patologías no-cáncer agrupadas)
- `Patient_group` para muestras `Malignant` (tipo de cáncer)

Además, se usa un **umbral sobre p(cáncer)** (`1 - p(nonMalignant)`) para controlar falsos negativos.

In [1]:
from __future__ import annotations
from pathlib import Path
from dataclasses import replace
import logging
import warnings
from sklearn.exceptions import ConvergenceWarning

import pandas as pd
import numpy as np

from time import perf_counter
from tqdm.auto import tqdm

from genomics_dl.models.train_multiclass import MulticlassTrainConfig, run_training

warnings.filterwarnings("ignore", category=ConvergenceWarning)

logging.getLogger("alembic").setLevel(logging.ERROR)
logging.getLogger("alembic.runtime.migration").setLevel(logging.ERROR)

logging.getLogger("mlflow").setLevel(logging.ERROR)
logging.getLogger("sqlalchemy").setLevel(logging.ERROR)
warnings.filterwarnings(
    "ignore",
    category=RuntimeWarning,
    message=".*invalid value encountered in divide.*",
)
warnings.filterwarnings(
    "ignore",
    category=RuntimeWarning,
    message=".*A worker stopped while some jobs were given to the executor.*",
)

/workspaces/TFM/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Rutas y carga de datos

In [2]:
# Paths
DATA_PROCESSED = Path("../data/processed")
TRAIN_PATH = DATA_PROCESSED / "gse183635_tep_tpm_train.parquet"
TEST_PATH  = DATA_PROCESSED / "gse183635_tep_tpm_test.parquet"

df_train = pd.read_parquet(TRAIN_PATH)
df_test  = pd.read_parquet(TEST_PATH)

df_train.shape, df_test.shape

((1880, 5452), (471, 5452))

### Separación genes vs metadatos

In [3]:
metadata_cols = [
    "Sample ID",
    "Patient_group",
    "Stage",
    "Sex",
    "Age",
    "Sample-supplying institution",
    "Training series",
    "Evaluation series",
    "Validation series",
    "lib.size",
    "classificationScoreCancer",
    "Class_group",
]

# Genes: columnas ENSG...
gene_cols = [c for c in df_train.columns if str(c).startswith("ENSG")]

assert "Class_group" in df_train.columns
assert "Patient_group" in df_train.columns
assert len(gene_cols) > 0
assert set(gene_cols).isdisjoint(set(metadata_cols))

len(gene_cols), gene_cols[:5]

(5440,
 ['ENSG00000000419',
  'ENSG00000000460',
  'ENSG00000000938',
  'ENSG00000001036',
  'ENSG00000001461'])

### Sanity check de etiquetas (train)

In [7]:
df_train[metadata_cols]

,Sample ID,Patient_group,Stage,Sex,Age,Sample-supplying institution,Training series,Evaluation series,Validation series,lib.size,classificationScoreCancer,Class_group
0,TR4021-OVA-Supernat,Endometrial cancer,II,F,71.0,Institute 2,0,1,0,411292,0.703062,Malignant
1,NKI-NSCLC-4987-TR2895,Non-small-cell lung cancer,IV,M,82.0,Institute 5,0,0,1,209670,0.990598,Malignant
2,Vumc-ORC-01-031-TR2368,Colorectal cancer,IV,M,68.0,Institute 13,0,0,1,426083,0.784636,Malignant
3,Vumc-HD-246-TR1916,Asymptomatic controls,n.a.,M,48.0,Institute 13,1,0,0,261776,0.899817,nonMalignant
4,TR4310-OVA-LUMC,Ovarian cancer,III,F,20.0,Institute 13,0,0,1,384785,0.871777,Malignant
...,...,...,...,...,...,...,...,...,...,...,...,...
1875,TR4241-HN-VUMC-BB,Head and neck cancer,IV,M,62.0,Institute 13,0,0,1,1051804,0.992232,Malignant
1876,TR3401-OVA-LUMC,Ovarian cancer,III,F,69.0,Institute 13,0,0,1,48182,0.955968,Malignant
1877,UMCG-NCSLC-1162-TR2886,Non-small-cell lung cancer,IV,M,77.0,Institute 9,0,0,1,91677,0.964765,Malignant
1878,TR4065-URO-RAD,Urothelial cancer,IV,M,77.0,Institute 8,0,0,1,624987,0.984430,Malignant


In [4]:
df_train["Class_group"].value_counts(dropna=False)

Class_group
Malignant       1302
nonMalignant     578
Name: count, dtype: int64

In [5]:
df_train.loc[df_train["Class_group"].astype(str) == "Malignant", "Patient_group"].value_counts().head(20)

Patient_group
Non-small-cell lung cancer    417
Ovarian cancer                114
Glioma                        113
Pancreatic cancer              93
Breast cancer                  80
Head and neck cancer           79
Cholangiocarcinoma             71
Colorectal cancer              69
Melanoma                       54
Sarcoma                        44
Endometrial cancer             34
Prostate cancer                23
Multiple Myeloma               22
Urothelial cancer              22
Renal cell cancer              20
Hepatocellular carcinoma       19
Lymphoma                       16
Esophageal carcinoma           12
Name: count, dtype: int64

## Experimentos baseline (sweep)

Ranking:
1) Minimizar `test_cancer_fn` (cáncer predicho como nonMalignant)
2) Maximizar `test_cancer_recall_sensitivity`
3) Maximizar `test_f1_macro`

In [6]:
def slugify_token(value):
    return str(value).replace(".", "p").replace("-", "m")

def build_model_name(clf_name, feat_cfg, malignant_weight, variant_tag):
    return "_".join([
        clf_name,
        f"pca{int(feat_cfg['use_pca'])}",
        f"log{int(feat_cfg['selector_on_log'])}",
        f"vq{int(feat_cfg['var_quantile']*100)}",
        f"mw{slugify_token(malignant_weight)}",
        variant_tag,
    ])

def fmt_secs(s: float) -> str:
    s = int(max(0, s))
    h = s // 3600
    m = (s % 3600) // 60
    ss = s % 60
    if h > 0:
        return f"{h:d}h {m:02d}m {ss:02d}s"
    if m > 0:
        return f"{m:d}m {ss:02d}s"
    return f"{ss:d}s"

In [8]:
# Base config
base_cfg = MulticlassTrainConfig(
    train_path=str(TRAIN_PATH),
    test_path=str(TEST_PATH),
    model_name="multiclass_optimized",
    model_version="v0.2.0",
    use_pca=False,
    var_quantile=0.2,
    selector_on_log=False,
    pca_var_threshold=0.9,
    cv_splits=8,
    min_cancer_recall_for_threshold=0.9,
    threshold_objective="specificity",
    experiment_name="gse183635_multiclass_optimized",
    save_local_bundle=False,
    save_plots=False,
)

# Sweep - OPTIMIZED CONFIGURATION
# Apply learnings from binary model (var_quantile=0.10, no PCA)
feat_grid = [
    # Binary model winner: Less aggressive filtering, no PCA
    dict(use_pca=False, selector_on_log=False, var_quantile=0.10, pca_var_threshold=0.9),
    dict(use_pca=False, selector_on_log=False, var_quantile=0.15, pca_var_threshold=0.9),
    
    # Original baseline (for comparison)
    dict(use_pca=False, selector_on_log=False, var_quantile=0.20, pca_var_threshold=0.9),
]

clf_grid = [
    # LogisticRegression: Vary regularization
    ("logreg", dict(solver="lbfgs", max_iter=8500, C=0.5)),
    ("logreg", dict(solver="lbfgs", max_iter=8500, C=1.0)),
    ("logreg", dict(solver="lbfgs", max_iter=8500, C=2.0)),
    
    # RandomForest: Reemplaza SGD (que fallaba con NaN)
    ("rf", dict(n_estimators=500, max_depth=None)),
    ("rf", dict(n_estimators=1000, max_depth=None)),
    ("rf", dict(n_estimators=800, max_depth=20)),
    
    # ExtraTrees: Current best performer, tune further
    ("extratrees", dict(n_estimators=1200, max_depth=None, min_samples_leaf=4)),
    ("extratrees", dict(n_estimators=1000, max_depth=None, min_samples_leaf=2)),
    ("extratrees", dict(n_estimators=800, max_depth=20)),
    ("extratrees", dict(n_estimators=800, max_depth=None)),
]

malignant_weights = [2.0, 3.0, 4.0, 6.0]

sweep = []
for feat_cfg in feat_grid:
    for clf_name, clf_params in clf_grid:
        for mw in malignant_weights:
            sweep.append(dict(feat_cfg=feat_cfg, clf_name=clf_name, clf_params=clf_params, mw=mw))

len(sweep)

120

In [9]:
results = []
errors = []

total = len(sweep)
start_all = perf_counter()

# Calculadora de tiempo por iteración y estimación con media móvil
ema = None
alpha = 0.25
done = 0

pbar = tqdm(sweep, total=total, desc="Sweep", unit="run")

for combo in pbar:
    t0 = perf_counter()

    feat_cfg = combo["feat_cfg"]
    clf_name = combo["clf_name"]
    clf_params = combo["clf_params"]
    mw = combo["mw"]

    model_name = build_model_name(clf_name, feat_cfg, mw, variant_tag="sweep")
    cfg = replace(
        base_cfg,
        model_name=model_name,
        clf_name=clf_name,
        clf_params=clf_params,
        malignant_weight=mw,
        use_pca=feat_cfg["use_pca"],
        selector_on_log=feat_cfg["selector_on_log"],
        var_quantile=feat_cfg["var_quantile"],
        pca_var_threshold=feat_cfg["pca_var_threshold"],
    )

    try:
        out = run_training(cfg, feature_cols=gene_cols)
        tm = out["test_metrics"]
        results.append({
            "model_name": model_name,
            "clf_name": clf_name,
            "mw": mw,
            "use_pca": feat_cfg["use_pca"],
            "selector_on_log": feat_cfg["selector_on_log"],
            "var_quantile": feat_cfg["var_quantile"],
            "test_cancer_fn": tm["cancer_fn"],
            "test_cancer_fnr": tm["cancer_fnr"],
            "test_cancer_recall": tm["cancer_recall_sensitivity"],
            "test_f1_macro": tm["f1_macro"],
            "test_accuracy": tm["accuracy"],
            "mlflow_run_id": out["mlflow_run_id"],
        })
    except Exception as e:
        errors.append({
            "model_name": model_name,
            "clf_name": clf_name,
            "mw": mw,
            "use_pca": feat_cfg["use_pca"],
            "selector_on_log": feat_cfg["selector_on_log"],
            "var_quantile": feat_cfg["var_quantile"],
            "error": repr(e),
        })

    dt = perf_counter() - t0
    ema = dt if ema is None else (alpha * dt + (1 - alpha) * ema)

    done += 1
    elapsed = perf_counter() - start_all
    remaining = (total - done) * (ema if ema is not None else 0.0)

    pbar.set_postfix({
        "last": fmt_secs(dt),
        "avg": fmt_secs(ema),
        "elapsed": fmt_secs(elapsed),
        "eta": fmt_secs(remaining),
        "ok": len(results),
        "err": len(errors),
    })

# DataFrames finales
res_df = (
    pd.DataFrame(results)
      .sort_values(["test_cancer_fn", "test_cancer_fnr", "test_cancer_recall", "test_f1_macro"],
                   ascending=[True, True, False, False])
      .reset_index(drop=True)
)

err_df = pd.DataFrame(errors).reset_index(drop=True)

print("OK:", len(res_df), "Errores:", len(err_df))

Sweep:  20%|██        | 24/120 [23:03<1:22:58, 51.86s/run, last=51s, avg=51s, elapsed=23m 03s, eta=1h 22m 56s, ok=24, err=0]    /workspaces/TFM/.venv/lib/python3.12/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
Sweep: 100%|██████████| 120/120 [1:34:29<00:00, 47.25s/run, last=27s, avg=28s, elapsed=1h 34m 30s, eta=0s, ok=120, err=0]         

OK: 120 Errores: 0


In [10]:
display(res_df)

,model_name,clf_name,mw,use_pca,selector_on_log,var_quantile,test_cancer_fn,test_cancer_fnr,test_cancer_recall,test_f1_macro,test_accuracy,mlflow_run_id
0,rf_pca0_log0_vq20_mw6p0_sweep,rf,6.0,False,False,0.20,37,0.113497,0.886503,0.203700,0.456476,33d042b1b24640a0a71448636d37046f
1,rf_pca0_log0_vq15_mw6p0_sweep,rf,6.0,False,False,0.15,37,0.113497,0.886503,0.198840,0.460722,7024c2960fb640ad8b848f95d6c61c58
2,rf_pca0_log0_vq20_mw2p0_sweep,rf,2.0,False,False,0.20,37,0.113497,0.886503,0.194955,0.467091,c7e430208b9a498e84a378ff0b1959d7
3,extratrees_pca0_log0_vq15_mw3p0_sweep,extratrees,3.0,False,False,0.15,37,0.113497,0.886503,0.184055,0.462845,dc8be9eb8bbb436c86e37c581d4059c5
4,rf_pca0_log0_vq10_mw2p0_sweep,rf,2.0,False,False,0.10,37,0.113497,0.886503,0.177443,0.460722,19bfd1ab9e1a406ca8f9ee3754b694be
...,...,...,...,...,...,...,...,...,...,...,...,...
115,logreg_pca0_log0_vq10_mw4p0_sweep,logreg,4.0,False,False,0.10,48,0.147239,0.852761,0.331158,0.543524,d0974bc1e0de4bc9bf2e5ea25caeec4e
116,rf_pca0_log0_vq10_mw4p0_sweep,rf,4.0,False,False,0.10,48,0.147239,0.852761,0.194778,0.462845,f9b31b74698d4a1ba5783e0b85629133
117,extratrees_pca0_log0_vq20_mw6p0_sweep,extratrees,6.0,False,False,0.20,48,0.147239,0.852761,0.191407,0.456476,7ccd6b1ecc6842e194dcc19997443640
118,logreg_pca0_log0_vq10_mw2p0_sweep,logreg,2.0,False,False,0.10,49,0.150307,0.849693,0.331765,0.539278,0de32f1c4fcc4e1db8e120afcd455621


In [11]:
if len(err_df) > 0:
    display(err_df)

## Entrenamiento final (guardar bundle en `models/`)

In [12]:
# Elegimos el mejor del sweep
best = res_df.iloc[0].to_dict()
best

{'model_name': 'rf_pca0_log0_vq20_mw6p0_sweep',
 'clf_name': 'rf',
 'mw': 6.0,
 'use_pca': False,
 'selector_on_log': False,
 'var_quantile': 0.2,
 'test_cancer_fn': 37,
 'test_cancer_fnr': 0.11349693251533742,
 'test_cancer_recall': 0.8865030674846626,
 'test_f1_macro': 0.2036997530188324,
 'test_accuracy': 0.4564755838641189,
 'mlflow_run_id': '33d042b1b24640a0a71448636d37046f'}

In [13]:
best_cfg = replace(
    base_cfg,
    model_name="multiclass_final",
    model_version="v0.2.0",
    clf_name=best["clf_name"],
    malignant_weight=float(best["mw"]),
    use_pca=bool(best["use_pca"]),
    selector_on_log=bool(best["selector_on_log"]),
    var_quantile=float(best["var_quantile"]),
    save_local_bundle=True,
    save_plots=True,
)

final_out = run_training(best_cfg, feature_cols=gene_cols)
final_out

{'mlflow_run_id': 'fad359bca714469a8beafb4aced3766e',
 'cv_metrics': {'accuracy': 0.4574468085106383,
  'balanced_accuracy': 0.18150743872417488,
  'f1_macro': 0.19490163735999158,
  'f1_weighted': 0.41450774357146863,
  'log_loss': 1.810715875202977,
  'cancer_threshold': 0.602,
  'cancer_tn': 321,
  'cancer_fp': 257,
  'cancer_fn': 125,
  'cancer_tp': 1177,
  'cancer_fnr': 0.09600614439324116,
  'cancer_recall_sensitivity': 0.9039938556067588,
  'cancer_specificity': 0.5553633217993079,
  'cancer_precision': 0.8207810320781032,
  'cancer_roc_auc': 0.8444102498684483,
  'cancer_pr_auc': 0.9184819014174062,
  'per_class_report': {'Breast cancer': {'precision': 0.49019607843137253,
    'recall': 0.3125,
    'f1-score': 0.3816793893129771,
    'support': 80.0},
   'Cholangiocarcinoma': {'precision': 0.375,
    'recall': 0.08450704225352113,
    'f1-score': 0.13793103448275862,
    'support': 71.0},
   'Colorectal cancer': {'precision': 0.38,
    'recall': 0.2753623188405797,
    'f1-scor